# DistilBERT: the whole record

One notebook that reproduces the complete Week 2 transformer work, like `06_export_baseline_model.ipynb` does for the baseline:

1. Leakage checks on the shared split (no `row_id` in two splits, near-duplicate openings)
2. Hyperparameter tuning on validation loss (optional, 4 runs)
3. Fine-tune `distilbert-base-uncased` on `raw_text` (optional: or load the saved model)
4. Evaluate on validation and test: accuracy, precision, recall, F1 (Fake = positive class), confusion matrix, speed
5. Predictions file for AI/ML-03 and a check against the official one
6. Comparison with the baseline and the one-word `(Reuters)` rule, confidence calibration, probes
7. Export the model folder (and 20 MB parts for downloads) for a website

**Where to run it**
- **Kaggle** (GPU on, Internet on, dataset `fake-detection` attached): training and tuning. `RUN_TRAINING` is on by default there.
- **Locally**: evaluation, comparison and export from the saved model in `models/transformer/distilbert/`. Training on a laptop CPU crashed before, so `RUN_TRAINING` is off by default.

**DistilBERT is not a pickle.** The model is a folder (`model.safetensors` plus config and tokenizer files). The website loads that folder with `transformers`.
Nothing official is overwritten: a local retrain goes to `models/transformer/export/retrained_model/`.

In [ ]:
import gc, hashlib, json, os, platform, re, time
from pathlib import Path

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # avoids a Windows OpenMP crash

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments, set_seed)

ON_KAGGLE = Path("/kaggle/input").exists()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---- switches -------------------------------------------------------------
RUN_TUNING = False            # 4 extra training runs, about 45 minutes on 2 Kaggle GPUs
RUN_TRAINING = ON_KAGGLE      # train the final model (about 13 minutes on 2 Kaggle GPUs); False = load the saved model
EVAL_SPLITS = ["val", "test"] # on a laptop CPU this takes about an hour; use ["test"] for about half
MAKE_PARTS = True             # also write 20 MB parts of model.safetensors (the Kaggle panel refused the big file)
SMOKE_ROWS = None             # e.g. 200 = tiny training run, only to check that the code works
# ---------------------------------------------------------------------------

SEED = 42
MODEL_NAME = "distilbert-base-uncased"
BASE_CONFIG = {"lr": 2e-5, "max_len": 256, "epochs": 2}   # the configuration chosen in Week 2

if ON_KAGGLE:
    DATA = Path("/kaggle/input/datasets/govindvijay001/fake-detection")
    ROOT, OFFICIAL_RES = None, None
    MASTER, SPLIT = DATA / "master.csv", DATA / "split.csv"
    MODEL_PATH = Path("/kaggle/working/distilbert_v1")
    OUT = Path("/kaggle/working/distilbert_export")
else:
    ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
    MASTER, SPLIT = ROOT / "data" / "interim" / "master.csv", ROOT / "data" / "splits" / "split.csv"
    MODEL_PATH = ROOT / "models" / "transformer" / "distilbert"
    OUT = ROOT / "models" / "transformer" / "export"
    OFFICIAL_RES = ROOT / "results"
OUT.mkdir(parents=True, exist_ok=True)

checks = {}   # summarised in the last cell
print("device:", DEVICE, "| GPUs:", torch.cuda.device_count(), "| torch", torch.__version__, "| transformers", transformers.__version__)
print("RUN_TRAINING:", RUN_TRAINING, "| RUN_TUNING:", RUN_TUNING, "| EVAL_SPLITS:", EVAL_SPLITS)
print("saved model folder:", MODEL_PATH)
print("outputs go to:", OUT)

## 1. Load the shared data and split

In [ ]:
split_df = pd.read_csv(SPLIT)
checks["every row_id is in exactly one split"] = bool(split_df["row_id"].is_unique)

df = pd.read_csv(MASTER, usecols=["row_id", "label", "text", "raw_text"]).merge(split_df, on="row_id", how="left")
assert df["split"].notna().all(), "some row_ids are missing from split.csv"
train, val, test = (df[df["split"] == s] for s in ("train", "val", "test"))
print(len(df), "articles | train", len(train), "| val", len(val), "| test", len(test))
print("Expected for the real data: 38656 | 27058 | 5799 | 5799")
print("labels (0 = Real, 1 = Fake):", df["label"].value_counts().to_dict())

## 2. Leakage check: near-duplicate openings

Exact duplicates were removed in `master.csv`. This compares the first 300 normalised characters of each body against the training set.
Earlier result: 91 of 5,799 validation and 77 of 5,799 test articles. That is too few to explain near-perfect scores, so the scores come from how separable the two sources are.

In [ ]:
norm = lambda t: re.sub(r"\W+", " ", str(t).lower()).strip()[:300]
train_keys = set(train["text"].map(norm))
overlap = {s: int(d["text"].map(norm).isin(train_keys).sum()) for s, d in (("val", val), ("test", test))}
print("articles whose opening matches a training article:", overlap)
if len(df) == 38656:
    checks["near-duplicate counts match the earlier check (val 91, test 77)"] = overlap == {"val": 91, "test": 77}
else:
    print("Not the full 38,656-article dataset: the comparison with the earlier counts (91 and 77) is skipped.")

## 3. Training helpers

Settings are identical to the Week 2 run: learning rate 2e-5, weight decay 0.01, batch 16 per GPU, 2 epochs, 256 tokens, fp16 on a GPU, seed 42, best validation F1 kept.
Only the train and validation splits are used. **The test split is never touched during training or tuning.**

In [ ]:
CKPT = OUT / "checkpoints"

def make_ds(tok, d, max_len):
    ds = Dataset.from_dict({"text": d["raw_text"].tolist(), "label": d["label"].tolist()})
    return ds.map(lambda b: tok(b["text"], truncation=True, max_length=max_len),
                  batched=True, remove_columns=["text"])

def train_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds),
            "precision": precision_score(p.label_ids, preds, zero_division=0),
            "recall": recall_score(p.label_ids, preds, zero_division=0),
            "f1": f1_score(p.label_ids, preds, zero_division=0)}

def run_training(lr, max_len, epochs, save_dir=None, batch_size=16):
    """Fine-tune once. With save_dir the best model and tokenizer are saved there."""
    set_seed(SEED)
    tr, va = train, val
    if SMOKE_ROWS:
        tr, va = train.sample(min(SMOKE_ROWS, len(train)), random_state=SEED), val.sample(min(max(SMOKE_ROWS // 2, 1), len(val)), random_state=SEED)
    tk = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds, val_ds = make_ds(tk, tr, max_len), make_ds(tk, va, max_len)
    mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
    args = TrainingArguments(
        output_dir=str(CKPT), num_train_epochs=epochs,
        per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
        learning_rate=lr, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch" if save_dir else "no", save_total_limit=1,
        load_best_model_at_end=bool(save_dir), metric_for_best_model="f1",
        logging_steps=100, fp16=torch.cuda.is_available(), seed=SEED, report_to="none")
    trainer = Trainer(model=mdl, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=DataCollatorWithPadding(tk), compute_metrics=train_metrics)
    t0 = time.time()
    trainer.train()
    minutes = round((time.time() - t0) / 60, 1)
    r = trainer.evaluate()
    log = pd.DataFrame(trainer.state.log_history)
    if save_dir:
        Path(save_dir).mkdir(parents=True, exist_ok=True)
        trainer.save_model(str(save_dir))
        tk.save_pretrained(str(save_dir))
        log.to_csv(OUT / "training_log.csv", index=False)
    row = {"lr": lr, "max_len": max_len, "epochs": epochs, "val_loss": r["eval_loss"],
           "val_acc": r["eval_accuracy"], "val_f1": r["eval_f1"], "minutes": minutes}
    del trainer, mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row, log

## 4. Hyperparameter tuning (optional, needs a GPU)

Four settings, selected by **validation loss** (accuracy and F1 are already at the ceiling, so they cannot separate the runs). The test split is not used.
In Week 2 the winner was learning rate 2e-5 at 256 tokens. Differences of about 1e-4 in loss are within run-to-run noise (one seed).

In [ ]:
BEST = dict(BASE_CONFIG)
if RUN_TUNING:
    assert DEVICE == "cuda" or SMOKE_ROWS, "tuning needs a GPU"
    rows = []
    for lr, ml in [(2e-5, 256), (3e-5, 256), (5e-5, 256), (2e-5, 128)]:
        print(f"\n===== lr {lr}, max_len {ml} =====")
        row, _ = run_training(lr, ml, BASE_CONFIG["epochs"])
        rows.append(row)
    tuning = pd.DataFrame(rows).sort_values("val_loss")
    tuning.to_csv(OUT / "tuning_results.csv", index=False)
    print(tuning.to_string(index=False))
    best_row = tuning.iloc[0]
    BEST = {"lr": float(best_row["lr"]), "max_len": int(best_row["max_len"]), "epochs": BASE_CONFIG["epochs"]}
    print("\nbest by validation loss:", BEST)
else:
    print("Tuning skipped (RUN_TUNING = False). Using the Week 2 configuration:", BEST)
    print("Week 2 results: lr 2e-5 / 256 tokens val loss 0.000154; 3e-5 / 256: 0.000261; 5e-5 / 256: 0.001225; 2e-5 / 128: 0.002232")

## 5. Train the final model, or load the saved one

On Kaggle the model is saved to `/kaggle/working/distilbert_v1`. **Kaggle deletes that folder when the session resets**, so download or export it straight away (section 10).
A local retrain never overwrites `models/transformer/distilbert/`.

In [ ]:
if RUN_TRAINING:
    assert DEVICE == "cuda" or SMOKE_ROWS, "full training needs a GPU. Run it on Kaggle, or set SMOKE_ROWS for a tiny test run."
    TRAIN_DIR = MODEL_PATH if ON_KAGGLE else OUT / "retrained_model"
    row, log = run_training(BEST["lr"], BEST["max_len"], BEST["epochs"], save_dir=TRAIN_DIR)
    MODEL_PATH = TRAIN_DIR
    print("\nvalidation after training:", {k: round(v, 6) if isinstance(v, float) else v for k, v in row.items()})
    print(log[log["eval_loss"].notna()][["epoch", "eval_loss", "eval_accuracy", "eval_precision", "eval_recall", "eval_f1"]].to_string(index=False))
    print("model saved to:", MODEL_PATH)
else:
    assert (MODEL_PATH / "model.safetensors").exists(), f"no saved model in {MODEL_PATH}"
    print("Loading the saved model from", MODEL_PATH, "(RUN_TRAINING = False)")
SCORE_MAX_LEN = BEST["max_len"] if RUN_TRAINING else 256

## 6. Evaluate on validation and test

Fake = positive class. Articles are sorted by length so batches need less padding. **The test split is scored once here and not used for any choice.**

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(DEVICE).eval()

def score_texts(texts, max_len=SCORE_MAX_LEN, batch_size=None):
    batch_size = batch_size or (64 if DEVICE == "cuda" else 32)
    order = np.argsort([len(t) for t in texts])
    probs = np.zeros(len(texts))
    t0 = time.time()
    with torch.no_grad():
        for s in range(0, len(order), batch_size):
            idx = order[s:s + batch_size]
            enc = tok([texts[i] for i in idx], truncation=True, max_length=max_len,
                      padding=True, return_tensors="pt").to(DEVICE)
            probs[idx] = torch.softmax(model(**enc).logits.float(), dim=1)[:, 1].cpu().numpy()
            if (s // batch_size) % 40 == 0:
                print(f"  {min(s + batch_size, len(order))}/{len(order)} articles | {time.time() - t0:.0f}s", flush=True)
    return probs, time.time() - t0

ev = df[df["split"].isin(EVAL_SPLITS)].reset_index(drop=True)
probs, secs = score_texts(ev["raw_text"].tolist())
predictions = pd.DataFrame({"row_id": ev["row_id"], "split": ev["split"], "y_true": ev["label"],
                            "y_pred": (probs >= 0.5).astype(int), "prob_fake": probs})
print(f"\nscored {len(ev)} articles in {secs:.0f}s on {DEVICE} ({secs / len(ev):.4f} s per article)")

def four_metrics(y, pred):
    return {"accuracy": round(accuracy_score(y, pred), 4),
            "precision_fake": round(precision_score(y, pred, zero_division=0), 4),
            "recall_fake": round(recall_score(y, pred, zero_division=0), 4),
            "f1_fake": round(f1_score(y, pred, zero_division=0), 4)}

metrics = {}
for sp in EVAL_SPLITS:
    d = predictions[predictions["split"] == sp]
    metrics[sp] = {**four_metrics(d["y_true"], d["y_pred"]),
                   "confusion_matrix": confusion_matrix(d["y_true"], d["y_pred"], labels=[0, 1]).tolist(),
                   "errors": int((d["y_true"] != d["y_pred"]).sum())}
    print(sp, metrics[sp])

if "test" in EVAL_SPLITS:
    d = predictions[predictions["split"] == "test"]
    print("\nTest classification report:")
    print(classification_report(d["y_true"], d["y_pred"], target_names=["Real", "Fake"], digits=4, zero_division=0))

RES_OUT = OUT / "results"          # not the official results/ folder
RES_OUT.mkdir(exist_ok=True)
predictions.to_csv(RES_OUT / "predictions_distilbert.csv", index=False)
metrics_json = {sp: {k: v for k, v in m.items() if k != "errors"} for sp, m in metrics.items()}
metrics_json.update({"device": DEVICE, "seconds_total": round(secs, 1), "seconds_per_article": round(secs / len(ev), 4)})
(RES_OUT / "metrics_distilbert.json").write_text(json.dumps(metrics_json, indent=2))
print("saved predictions_distilbert.csv and metrics_distilbert.json in", RES_OUT)

## 7. Compare with the official files

In [ ]:
official_pred = OFFICIAL_RES / "transformer" / "predictions_distilbert.csv" if OFFICIAL_RES else None
if official_pred is not None and official_pred.exists():
    off = pd.read_csv(official_pred)
    m = predictions.merge(off, on="row_id", suffixes=("", "_off"))
    agree = float((m["y_pred"] == m["y_pred_off"]).mean())
    gap = float(np.abs(m["prob_fake"] - m["prob_fake_off"]).max())
    print(f"articles in common: {len(m)} | same label as predictions_distilbert.csv: {agree:.4f} | largest probability gap: {gap:.2e}")
    if RUN_TRAINING:
        print("This is a retrained model, so small differences from the official file are normal (GPU training is not perfectly deterministic).")
    else:
        checks["predictions match predictions_distilbert.csv"] = bool(len(m) == len(predictions) and agree == 1.0 and gap < 1e-3)
else:
    print("Official predictions_distilbert.csv not available here: skipped.")

official_metrics = OFFICIAL_RES / "transformer" / "metrics_distilbert.json" if OFFICIAL_RES else None
if official_metrics is not None and official_metrics.exists() and not RUN_TRAINING:
    om = json.loads(official_metrics.read_text())
    keys = ["accuracy", "precision_fake", "recall_fake", "f1_fake"]
    shared = [sp for sp in EVAL_SPLITS if sp in om]
    checks["metrics match metrics_distilbert.json"] = bool(shared) and all(
        abs(metrics[sp][k] - om[sp][k]) < 1e-4 for sp in shared for k in keys)
    print("metrics match the official metrics_distilbert.json:", checks["metrics match metrics_distilbert.json"])

## 8. Baseline and rule comparison, errors, calibration, probes

Runs if the baseline predictions (`results/baseline/predictions_v0_raw.csv`) are available. The rule predicts Real if the article contains `(Reuters)`, otherwise Fake.

In [ ]:
t = predictions[predictions["split"] == "test"].copy()
base_file = OFFICIAL_RES / "baseline" / "predictions_v0_raw.csv" if OFFICIAL_RES else None
if len(t) and base_file is not None and base_file.exists():
    base = pd.read_csv(base_file)
    base = base[base["split"] == "test"].set_index("row_id")
    t = t.set_index("row_id").sort_index()
    base = base.loc[t.index]
    text = df.set_index("row_id").loc[t.index, "raw_text"]
    rule = (~text.str.contains("(Reuters)", regex=False)).astype(int)
    rows = []
    for name, pred in [("Rule: '(Reuters)' = Real, else Fake", rule),
                       ("TF-IDF + Logistic Regression", base["y_pred"]),
                       ("DistilBERT (this notebook)", t["y_pred"])]:
        tn, fp, fn, tp = confusion_matrix(t["y_true"], pred, labels=[0, 1]).ravel()
        rows.append({"model": name, **four_metrics(t["y_true"], pred), "FP": fp, "FN": fn, "errors": fp + fn})
    print(pd.DataFrame(rows).to_string(index=False))
    b_err, d_err = base["y_pred"] != t["y_true"], t["y_pred"] != t["y_true"]
    print(f"\nbaseline errors fixed by DistilBERT: {int((b_err & ~d_err).sum())} | errors only DistilBERT makes: {int((d_err & ~b_err).sum())}")
    t = t.reset_index()
else:
    print("Baseline predictions not available here: comparison skipped.")

if len(t):
    wrong = t[t["y_true"] != t["y_pred"]]
    print("\nDistilBERT test errors:", len(wrong))
    if len(wrong):
        w = wrong.merge(df[["row_id", "raw_text"]], on="row_id").head(5)
        for _, r in w.iterrows():
            print(f"  row {r['row_id']} true={r['y_true']} prob_fake={r['prob_fake']:.3f} | {r['raw_text'][:90]}")
    t["confidence"] = t["prob_fake"].apply(lambda p: max(p, 1 - p))
    t["correct"] = t["y_true"] == t["y_pred"]
    t["bucket"] = pd.cut(t["confidence"], [0.5, 0.7, 0.9, 0.99, 1.0])
    print("\nAccuracy inside each confidence range (test):")
    print(t.groupby("bucket", observed=True)["correct"].agg(["mean", "count"]))

In [ ]:
def predict_one(text, max_len=SCORE_MAX_LEN):
    enc = tok(text, truncation=True, max_length=max_len, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        p = torch.softmax(model(**enc).logits.float(), dim=1)[0, 1].item()
    return ("Fake" if p >= 0.5 else "Real"), round(max(p, 1 - p), 4)

probes = {   # hand-written, NOT dataset articles: they show what the model reads
    "false claim, Reuters style": "WASHINGTON (Reuters) - The U.S. government confirmed on Tuesday that the moon landing was staged, officials said.",
    "true-sounding, blog style": "Watch this video: the Senate passed the budget bill today. Featured image via Twitter. Read more!",
    "neutral, no source cues": "The city council will meet next week to discuss the new parking rules.",
    "one word": "Short.",
}
base_bundle = None
if ROOT is not None and (ROOT / "models" / "baseline" / "export" / "fake_news_baseline.pkl").exists():
    import pickle
    with open(ROOT / "models" / "baseline" / "export" / "fake_news_baseline.pkl", "rb") as f:
        base_bundle = pickle.load(f)
for name, text in probes.items():
    line = f"{name:28s} | distilbert: {predict_one(text)}"
    if base_bundle is not None:
        pb = float(base_bundle["pipeline"].predict_proba([text])[0, 1])
        line += f" | baseline: {('Fake' if pb >= 0.5 else 'Real', round(max(pb, 1 - pb), 4))}"
    print(line)
print("\nWeek 2 finding: both models read source style (a false Reuters-style claim comes out Real, blog-style text comes out Fake).")

## 9. Speed

Single-article latency (what a website sees per click). CPU and GPU numbers are **not comparable**: record the device with the number.
Week 2 reference: about 0.36 s per article on a laptop CPU in batches of 32.

In [ ]:
sample = test["raw_text"].head(20).tolist()
t0 = time.time()
for s in sample:
    predict_one(s)
latency = (time.time() - t0) / len(sample)
print(f"single-article latency: {latency * 1000:.0f} ms on {DEVICE}")
print(f"batch scoring: {secs / len(ev):.4f} s per article on {DEVICE}")

## 10. Export for a website

The export is a **folder**: `model.safetensors`, `config.json`, `tokenizer.json`, `tokenizer_config.json`, plus `export_info.json` (label map, input format, library versions, checksum, test metrics).
A SHA256 checksum lets you prove a downloaded copy is identical. For downloads that fail, 20 MB parts can be joined again (instructions below).

In [ ]:
EXPORT_MODEL = OUT / "distilbert_model"
model.save_pretrained(EXPORT_MODEL)
tok.save_pretrained(EXPORT_MODEL)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

weights = EXPORT_MODEL / "model.safetensors"
info = {
    "base_model": MODEL_NAME, "labels": {"0": "Real", "1": "Fake"}, "threshold": 0.5,
    "max_len": SCORE_MAX_LEN, "input_format": "title + '. ' + body, one string (same as raw_text), no cleaning",
    "transformers_version": transformers.__version__, "torch_version": torch.__version__,
    "weights_bytes": weights.stat().st_size, "weights_sha256": sha256(weights),
    "test_metrics": metrics.get("test"),
}
(OUT / "export_info.json").write_text(json.dumps(info, indent=2))
print("export folder:", EXPORT_MODEL)
for p in sorted(EXPORT_MODEL.iterdir()):
    print(f"  {p.name:25s} {p.stat().st_size / 1e6:8.1f} MB")
print("weights sha256:", info["weights_sha256"])

if MAKE_PARTS:
    parts_dir = OUT / "parts"
    parts_dir.mkdir(exist_ok=True)
    for old in parts_dir.glob("model_part*"):
        old.unlink()
    n = 0
    with open(weights, "rb") as f:
        while chunk := f.read(20 * 1024 * 1024):
            (parts_dir / f"model_part{n:02d}").write_bytes(chunk)
            n += 1
    print(f"\n{n} parts of 20 MB in {parts_dir}")
    print("To rejoin on Windows (Command Prompt, in the folder with all parts):  copy /b model_part* model.safetensors")
    print("Then check the size (", info["weights_bytes"], "bytes ) and the SHA256 above.")

## 11. How the website uses it

Copy these two functions into the website's code and point `load_distilbert` at the model folder. It needs `torch` and `transformers`.
The output has the same format as `src/predictor.py` (`label`, `confidence`, `prob_fake`). Keep the confidence caution: it is not accuracy, and the model reads source style.

In [ ]:
def load_distilbert(folder, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    tk = AutoTokenizer.from_pretrained(folder)
    mdl = AutoModelForSequenceClassification.from_pretrained(folder).to(device).eval()
    return tk, mdl, device

def predict_text(tk, mdl, device, text, max_len=256):
    if not isinstance(text, str) or not text.strip():
        return {"label": "Unknown", "confidence": None, "prob_fake": None, "error": "Empty input"}
    enc = tk(text, truncation=True, max_length=max_len, return_tensors="pt").to(device)
    with torch.no_grad():
        p = float(torch.softmax(mdl(**enc).logits.float(), dim=1)[0, 1])
    out = {"label": "Fake" if p >= 0.5 else "Real", "confidence": max(p, 1 - p), "prob_fake": p}
    if len(text.split()) < 30:
        out["warning"] = "Very short text. The model was trained on full news articles, so this result is unreliable."
    return out

tk2, mdl2, dev2 = load_distilbert(EXPORT_MODEL)
for text in [test["raw_text"].iloc[0], "Watch this video, featured image via Twitter. Read more!", ""]:
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in predict_text(tk2, mdl2, dev2, text).items()})

check_texts = test["raw_text"].head(20).tolist()
enc = tk2(check_texts, truncation=True, max_length=SCORE_MAX_LEN, padding=True, return_tensors="pt").to(dev2)
with torch.no_grad():
    p_exp = torch.softmax(mdl2(**enc).logits.float(), dim=1)[:, 1].cpu().numpy()
p_mem, _ = score_texts(check_texts)
checks["exported folder reloads with the same predictions"] = bool(np.abs(p_exp - p_mem).max() < 1e-3)

## 12. Summary of the checks

In [ ]:
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "|", name)
print()
for sp, m in metrics.items():
    print(sp, {k: v for k, v in m.items() if k != "confusion_matrix"}, "| confusion matrix", m["confusion_matrix"])
print("Reference (Week 2, official): val and test accuracy 1.0 with 0 errors; 0.36 s per article on a laptop CPU")

Where to run it

On Kaggle (training):

Attach the fake-detection dataset, and turn on the GPUs and Internet.
RUN_TRAINING is on by default there, so run it from top to bottom. Training takes about 13 minutes.
Set RUN_TUNING = True only if you want the four tuning runs, which take about 45 minutes more.
Kaggle deletes /kaggle/working when the session resets, so download the parts and export_info.json straight away. Join them as before (copy /b model_part* model.safetensors) and check the size and the SHA256.

Locally (evaluation and export from the saved model):

Open it in VS Code with the venv kernel, run from the notebooks\ folder. RUN_TRAINING is off by default, because training on your laptop CPU crashed earlier.
Scoring on a CPU takes about 35 minutes for ["test"] and about an hour for both splits. Set EVAL_SPLITS = ["test"] in the first cell if you want it quicker.
Done when the last cell prints PASS for the checks that apply. On the real data these should be:
every row_id is in exactly one split
near-duplicate counts match the earlier check (91 and 77)
predictions match predictions_distilbert.csv
metrics match metrics_distilbert.json
the exported folder reloads with the same predictions
Test accuracy should be 1.0 with zero errors, as in Week 2.

It overwrites nothing official. Local outputs go to models\transformer\export\ (gitignored), and a local retrain goes to export\retrained_model\, so models\transformer\distilbert\ is never touched.

What to expect
If you retrain on Kaggle, expect results close to Week 2 but not identical, because GPU training isn’t perfectly deterministic. The notebook says so when it compares against the official file.
The tuning table is regenerated from scratch with a fresh run. The old first row in your tuning_results.csv came from the lost model.
The calibration table will probably show everything in the top confidence bucket, since the model is nearly always sure. That is the overconfidence finding, not a bug.